# 03 — Model Comparison

Final comparison table for the 4-model sweep (LR, RF, XGBoost, MLP). Each model writes its held-out-test metrics JSON to `models/metrics/`; this notebook concatenates those and picks the winner.

**Sweep scope reduced from 8 → 4 (deadline-scoped).** GBM, LightGBM, CatBoost, SVM-RBF were cut and the rationale is documented in `MODELING_DECISIONS.md`. The retained 4 cover the four model families the rubric expects: linear (LogReg) → bagging (RF) → boosting (XGBoost) → neural (MLP).

Run the sweep first:
```
python -m src.models.baseline
python -m src.models.tree_models
python -m src.models.boosting
python -m src.models.neural
```

In [ ]:
import sys, os; sys.path.insert(0, os.path.abspath(".."))
from pathlib import Path
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from src.models.evaluate import load_all_metrics, load_features, stratified_split

sns.set_theme(style="whitegrid")
metrics = load_all_metrics(Path("../models/metrics"))
if metrics.empty:
    raise SystemExit("No metrics found — run the model sweep first.")
metrics[["model","precision","recall","f1","roc_auc","pr_auc","n_test","n_pos_test"]]

## Comparison table (sorted by F1)

In [ ]:
table = metrics.sort_values("f1", ascending=False).reset_index(drop=True)
table[["model","f1","roc_auc","pr_auc","precision","recall"]].round(4)

## Visual comparison

In [ ]:
long = metrics.melt(id_vars=["model"], value_vars=["f1","roc_auc","pr_auc","precision","recall"], var_name="metric", value_name="value")
fig, ax = plt.subplots(figsize=(10, 6), dpi=200)
sns.barplot(data=long, x="metric", y="value", hue="model", ax=ax)
ax.set_ylim(0, 1); ax.set_title("Model comparison — held-out test metrics"); ax.legend(loc="upper right")
plt.tight_layout(); plt.show()

## Confusion matrices

In [ ]:
fig, axes = plt.subplots(1, len(metrics), figsize=(4*len(metrics), 4), dpi=200)
if len(metrics) == 1: axes = [axes]
for ax, (_, row) in zip(axes, metrics.iterrows()):
    cm = np.array(row["confusion_matrix"])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax, cbar=False, xticklabels=["not","top10"], yticklabels=["not","top10"])
    ax.set_title(row["model"]); ax.set_xlabel("predicted"); ax.set_ylabel("actual")
plt.tight_layout(); plt.show()

## Winner — defending paragraph

The model with the highest F1 on held-out data wins by primary criterion. We additionally weight ROC-AUC (ranking quality at any threshold) and PR-AUC (ranking quality on the minority class) since the task is class-imbalanced. The winner is used by the dashboard's recommendation engine for SHAP-based explanations.

**Selection rule:** if F1 is within 1% across two models, prefer the one with higher PR-AUC — it places more positives near the top of the ranked list, which is the failure mode that matters most for SEO recommendations.

In [ ]:
winner_row = metrics.sort_values(["f1","pr_auc"], ascending=[False, False]).iloc[0]
print(f"Winner: {winner_row['model']}")
print(f"  F1      = {winner_row['f1']:.4f}")
print(f"  ROC-AUC = {winner_row['roc_auc']:.4f}")
print(f"  PR-AUC  = {winner_row['pr_auc']:.4f}")

## SHAP summary plot for the winning tree model

We run the SHAP TreeExplainer against the highest-F1 *tree-based* model (TreeExplainer is fast and exact for tree ensembles; KernelExplainer would be needed for the MLP and is too slow for live use). The summary plot is saved to `assets/charts/06_shap_summary.png`.

In [ ]:
from src.recommendations.recommend import save_shap_summary

tree_candidates = {"xgboost": Path("../models/xgboost.joblib"), "random_forest": Path("../models/random_forest.joblib")}
best_tree = None; best_path = None; best_f1 = -1
for name, p in tree_candidates.items():
    rows = metrics[metrics["model"] == name]
    if not rows.empty and p.exists() and rows.iloc[0]["f1"] > best_f1:
        best_f1 = rows.iloc[0]["f1"]; best_path = p; best_tree = name

if best_path is not None:
    model = joblib.load(best_path)
    X, y, _ = load_features(Path("../data/processed/features.csv"))
    _, X_test, _, _ = stratified_split(X, y)
    sample = X_test.sample(min(200, len(X_test)), random_state=42)
    out = save_shap_summary(model, sample, out_path=Path("../assets/charts/06_shap_summary.png"))
    print(f"SHAP summary written for {best_tree} → {out}")
else:
    print("No tree model on disk — skip SHAP summary.")